# Core utilities

> General helpers with stable-worldmodel's names: nested dicts, defaults, videos of dataset episodes.

In [ ]:
#| default_exp utils.core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
# Adapted from stable-worldmodel's utils (https://github.com/galilai-group/stable-worldmodel):
#
# MIT License
#
# Copyright (c) 2026 GalilAI-group
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

from __future__ import annotations

import logging
import os
from pathlib import Path
from typing import Any, Iterable

import numpy as np

logger = logging.getLogger(__name__)

DEFAULT_CACHE_DIR = os.path.expanduser('~/.stable_marl')   # overridden by $STABLEMARL_HOME (see data.get_cache_dir)
HF_BASE_URL = 'https://huggingface.co'

In [ ]:
#| export
def exists(val: Any) -> bool:
    "Whether `val` is not None."
    return val is not None


def default(val: Any, d: Any) -> Any:
    "`val` if it is not None, else `d`."
    return val if exists(val) else d


def flatten_dict(d: dict, parent_key: str = '', sep: str = '.') -> dict:
    "A nested dict as one level, keys joined by `sep` (e.g. ``{'a': {'b': 1}}`` -> ``{'a.b': 1}``)."
    items = {}
    for k, v in d.items():
        key = f'{parent_key}{sep}{k}' if parent_key else k
        if isinstance(v, dict):
            items.update(flatten_dict(v, key, sep=sep))
        else:
            items[key] = v
    return items


def get_in(mapping: Any, path: Iterable[str]) -> Any:
    "The value at the end of `path` in nested mappings, e.g. ``get_in(space, ['wall', 'color'])``."
    current = mapping
    for key in list(path):
        current = current[key]
    return current

## Videos of dataset episodes

`record_video_from_dataset` writes one MP4 per episode from image columns. A column can be
a single image per step, ``(T, H, W, C)`` (or channel-first, as datasets return 4-d images), or
one per agent, ``(T, num_agents, H, W, C)`` like ``pixels``: the agents are placed side by
side. Several views are stacked vertically (narrower ones padded with black).

In [ ]:
#| export
def _frames_hwc(x) -> np.ndarray:
    "A view as ``(T, H, W, C)`` uint8 frames: channel-first images transposed, agents side by side."
    arr = x.detach().cpu().numpy() if type(x).__module__.startswith('torch') else np.asarray(x)
    if arr.ndim == 4 and arr.shape[1] in (1, 3) and arr.shape[-1] not in (1, 3):
        arr = arr.transpose(0, 2, 3, 1)
    if arr.ndim == 5:                                       # (T, agents, H, W, C) -> (T, H, agents * W, C)
        arr = np.concatenate(list(arr.transpose(1, 0, 2, 3, 4)), axis=2)
    if arr.ndim != 4:
        raise ValueError(f"expected image frames (T, H, W, C) or (T, num_agents, H, W, C), got shape {arr.shape}")
    if arr.dtype.kind == 'f':
        arr = (np.clip(arr, 0.0, 1.0) * 255).astype(np.uint8)
    return np.repeat(arr, 3, axis=-1) if arr.shape[-1] == 1 else arr


def record_video_from_dataset(
    video_path: str | Path,
    dataset: Any,
    episode_idx: int | list[int],
    max_steps: int = 500,
    fps: int = 30,
    viewname: str | list[str] = 'pixels') -> list[Path]:
    """
    Save episodes of `dataset` as ``<video_path>/episode_<i>.mp4``, from the image column(s)
    `viewname` (first `max_steps` steps). Returns the files written. Needs ``imageio`` with ffmpeg.
    """
    import imageio

    episode_idx = [episode_idx] if isinstance(episode_idx, (int, np.integer)) else list(episode_idx)
    viewname = [viewname] if isinstance(viewname, str) else list(viewname)
    missing = [v for v in viewname if v not in dataset.column_names]
    assert not missing, f"Some views in {viewname} are not in dataset key names {dataset.column_names}"
    Path(video_path).mkdir(parents=True, exist_ok=True)
    files = []
    for ep in episode_idx:
        steps = dataset.load_episode(int(ep))
        views = [_frames_hwc(steps[v])[:max_steps] for v in viewname]
        width = max(v.shape[2] for v in views)
        views = [np.pad(v, ((0, 0), (0, 0), (0, width - v.shape[2]), (0, 0))) for v in views]
        file = Path(video_path, f'episode_{ep}.mp4')
        imageio.mimsave(file, np.concatenate(views, axis=1), fps=fps)
        files.append(file)
    logger.info(f"Video saved to {video_path}")
    return files

### Tests

In [ ]:
import tempfile, torch
from fastcore.test import test_fail

assert flatten_dict({'a': {'b': 1, 'c': {'d': 2}}, 'e': 3}) == {'a.b': 1, 'a.c.d': 2, 'e': 3}
assert flatten_dict({'a': {'b': 1}}, sep='/') == {'a/b': 1}
assert default(None, 4) == 4 and default(0, 4) == 0 and exists(0) and not exists(None)
assert get_in({'wall': {'color': 'red'}}, ['wall', 'color']) == 'red'

# frames: per-agent views side by side, channel-first images back to channel-last, float images to uint8
agents = np.zeros((5, 2, 8, 6, 3), np.uint8); agents[:, 1] = 255
f = _frames_hwc(agents)
assert f.shape == (5, 8, 12, 3) and (f[:, :, :6] == 0).all() and (f[:, :, 6:] == 255).all()
assert _frames_hwc(torch.rand(5, 3, 8, 6)).shape == (5, 8, 6, 3) and _frames_hwc(torch.rand(5, 3, 8, 6)).dtype == np.uint8
test_fail(lambda: _frames_hwc(np.zeros((5, 2))), contains='expected image frames')

import stable_marl as sm
with tempfile.TemporaryDirectory() as tmp:
    world = sm.World('MultiGrid-FindGoal-15x15-v0', num_envs=1, agents=2, size=7, num_obstacles=0, n_clutter=0,
                     min_goal_spawn_distance=1, tile_size=4, max_episode_steps=6, add_pixels=True, render_mode='rgb_array')
    world.set_policy(sm.RandomPolicy(seed=0))
    world.collect(f'{tmp}/d.h5', episodes=2, seed=0, progress=False)
    ds = sm.HDF5Dataset(path=f'{tmp}/d.h5')
    files = record_video_from_dataset(f'{tmp}/videos', ds, [0, 1], viewname=['pixels', 'render'], fps=5)
    assert [p.name for p in files] == ['episode_0.mp4', 'episode_1.mp4'] and all(p.stat().st_size > 0 for p in files)
    import imageio
    frames = imageio.mimread(files[0])
    assert len(frames) == ds.lengths[0]
    test_fail(lambda: record_video_from_dataset(f'{tmp}/v', ds, 0, viewname='nope'), contains='nope')
    ds.close()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()